# Alpha Tuning Workbench

This notebook runs iterative alpha search against the WorldQuant API and ranks candidates by composite score with Sharpe priority.

Goal: find submittable candidates and push toward Sharpe >= 2.0.

In [15]:
import importlib
import json
import alpha_tuner

importlib.reload(alpha_tuner)
from alpha_tuner import run_tuning, as_dict

In [16]:
# Adjust these values per run
ITERATIONS = 2
BATCH_SIZE = 10
TARGET_SHARPE = 2.0
PAUSE_SEC = 0.75
SEED = 42
MAX_WORKERS = 3

In [17]:
ranked = run_tuning(
    iterations=ITERATIONS,
    batch_size=BATCH_SIZE,
    target_sharpe=TARGET_SHARPE,
    pause_sec=PAUSE_SEC,
    seed=SEED,
    max_workers=MAX_WORKERS,
)

top = ranked[:10]
for i, r in enumerate(top, start=1):
    print(f"{i:02d}. sharpe={r.sharpe} fitness={r.fitness} turnover={r.turnover} score={r.score:.3f} alpha={r.alpha_id}")
    print(f"    {r.expression}")


=== Round 1/2 | candidates=10 ===
Running in parallel with max_workers=3
Auth attempt -> url=https://api.worldquantbrain.com/authentication, login=mu************************om, password_len=15
Auth attempt -> url=https://api.worldquantbrain.com/authentication, login=mu************************om, password_len=15
Auth attempt -> url=https://api.worldquantbrain.com/authentication, login=mu************************om, password_len=15
Auth accepted for login=mu************************om
Auth accepted for login=mu************************om
Auth accepted for login=mu************************om
Rate limited (429). Waiting 2.0s before retrying POST https://api.worldquantbrain.com/simulations...
Rate limited (429). Waiting 4.0s before retrying POST https://api.worldquantbrain.com/simulations...
Rate limited (429). Waiting 8.0s before retrying POST https://api.worldquantbrain.com/simulations...
Auth attempt -> url=https://api.worldquantbrain.com/authentication, login=mu************************om, 

In [ ]:
rows = [as_dict(r) for r in ranked]
with open('alpha_tuning_results_notebook.json', 'w', encoding='utf-8') as f:
    json.dump(rows, f, indent=2)

print(f"Saved {len(rows)} rows to alpha_tuning_results_notebook.json")

## Single Alpha Quick Test

Paste one expression and evaluate it directly (simulation + alpha metrics).

In [ ]:
from alpha_tuner import evaluate_expression
from worldquant_api_starter import load_env_file, env, make_session, authenticate

expr = "rank(-ts_delta(close, 5)) * rank(ts_mean(volume, 20)) * rank(-ts_std_dev(close, 20)) * rank(ts_mean(close, 50) - ts_mean(close, 200))"

load_env_file()
base_url = env("WQ_BASE_URL", "https://api.worldquantbrain.com")
session = make_session()
authenticate(session, base_url)

run = evaluate_expression(session, base_url, expr)
print("expression:", run.expression)
print("simulation_id:", run.simulation_id)
print("alpha_id:", run.alpha_id)
print("sharpe:", run.sharpe)
print("fitness:", run.fitness)
print("turnover:", run.turnover)
print("returns:", run.returns)
print("drawdown:", run.drawdown)
print("score:", run.score)

## Fitness Push Pack

Runs a focused list of candidates around the current best signal and prints ranking by Fitness.

In [ ]:
from alpha_tuner import evaluate_expression

candidates = [
    "rank(ts_mean((vwap - close) / close, 3))",
    "group_neutralize(rank(ts_mean((vwap - close) / close, 3)), subindustry)",
    "rank(ts_mean((vwap - close) / close, 2))",
    "rank(ts_mean((vwap - close) / close, 8))",
    "ts_mean((vwap - close) / close, 3)",
    "rank(ts_zscore(ts_mean((vwap - close) / close, 5), 20))",
]

load_env_file()
base_url = env("WQ_BASE_URL", "https://api.worldquantbrain.com")
session = make_session()
authenticate(session, base_url)

runs = []
for idx, expr in enumerate(candidates, start=1):
    print(f"[{idx}/{len(candidates)}] {expr}")
    try:
        run = evaluate_expression(session, base_url, expr)
        runs.append(run)
        print(
            f"  -> sharpe={run.sharpe} fitness={run.fitness} turnover={run.turnover} "
            f"returns={run.returns} drawdown={run.drawdown} alpha={run.alpha_id}"
        )
    except Exception as exc:
        print(f"  -> failed: {exc}")

print("\nTop by fitness:")
for r in sorted(runs, key=lambda x: (x.fitness is not None, x.fitness or -999), reverse=True):
    print(f"fitness={r.fitness} sharpe={r.sharpe} turnover={r.turnover} alpha={r.alpha_id}")
    print(f"    {r.expression}")